### Assignment
Options pricing & trade structuring


* Load AAPL 18-Jun-2026 option prices as of 10-Apr (data from CBOE)
* Apply Put-Call Parity relationships to infer implied spot (compare with actual spot ref S0)
* Solve for implied vols using Black-Scholes model
* Calculated Deltas
* Rank Call Spreads and Put Spreads

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.optimize import brentq

import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Load inputs and the quote sheet

Strikes down the middle, call price mids in one column, put mids in the other. On a real desk you’d also store bid/ask, volume, open interest. This is a very simplified version.

In [2]:

# pricing date
valuation_date = pd.Timestamp("2026-04-10")
# option expiry date
expiry_date = pd.Timestamp("2026-06-18")
# time to expiry in years
T = (expiry_date - valuation_date).days / 365.25
# risk-free rate - SOFR: 3.70%
r = 0.037
# dividend yield: 1%
q = 0.01

# AAPL options prices data
# spot price
S0 = 261.48
data = {
    "Calls": [
        63.99, 58.95, 54.35, 49.74, 45.18, 40.74, 36.39, 32.19, 28.13, 24.25,
        20.6, 17.2, 14.12, 11.36, 8.95, 6.95, 5.28, 3.94, 2.91, 2.13, 1.55,
        1.13, 0.83, 0.61, 0.46,
    ],
    "Strike": list(range(200, 325, 5)),
    "Puts": [
        0.95, 1.17, 1.46, 1.81, 2.22, 2.73, 3.35, 4.11, 5.02, 6.11, 7.46,
        9.03, 10.94, 13.18, 15.84, 18.89, 22.3, 26.07, 30.09, 34.34, 38.94,
        43.68, 48.51, 53.41, 58.37,
    ],
}

option_prices = pd.DataFrame(data)
option_prices.set_index("Strike")

,Calls,Puts
Strike,,
200,63.99,0.95
205,58.95,1.17
210,54.35,1.46
215,49.74,1.81
220,45.18,2.22
225,40.74,2.73
230,36.39,3.35
235,32.19,4.11
240,28.13,5.02


### Put–call parity and inferred spot

Any strike should imply roughly the same spot/forward if the strip is clean. We use the **median** parity-implied level as a single `S_median`.


In [3]:
option_prices["S_parity"] = option_prices["Calls"] - option_prices["Puts"] + option_prices["Strike"] * np.exp(-r * T)
S_median = float(option_prices["S_parity"].median())
print("S0 parity median", S_median, "parity range", option_prices["S_parity"].min(), option_prices["S_parity"].max())


S0 parity median 261.35210085322103 parity range 259.86108425868656 261.6469276616791



Black–Scholes pricer from market inputs and option type/expiry/strike

In [4]:
def bs_price(S, K, T, r, q, sigma, call: bool) -> float:
    # Black-Scholes formula to calculate the price of a call or put option
    # assumption of constant dividend yield q
    if T <= 0:
        return float(max(S - K, 0.0)) if call else float(max(K - S, 0.0))
    v = sigma * np.sqrt(T)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / v
    d2 = d1 - v
    dfq = np.exp(-q * T)
    drf = np.exp(-r * T)
    if call:
        return float(S * dfq * norm.cdf(d1) - K * drf * norm.cdf(d2))
    return float(K * drf * norm.cdf(-d2) - S * dfq * norm.cdf(-d1))


Function to calculate the Black-Scholes delta of an option 

In [5]:
def bs_delta(S, K, T, r, q, sigma, call: bool) -> float:
    if T <= 0:
        if call:
            return 1.0 if S > K else 0.0 if S < K else 0.5
        return -1.0 if S < K else 0.0 if S > K else -0.5
    v = sigma * np.sqrt(T)
    if v <= 1e-14:
        dfq = np.exp(-q * T)
        k0 = K * np.exp(-(r - q) * T)
        if call:
            return float(dfq) if S > k0 else 0.0 if S < k0 else 0.5 * dfq
        return float(-dfq) if S < k0 else 0.0 if S > k0 else -0.5 * dfq
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * T) / v
    dfq = np.exp(-q * T)
    if call:
        return float(dfq * norm.cdf(d1))
    return float(dfq * (norm.cdf(d1) - 1.0))


Function to solve for the implied vol of an option given it price

In [6]:

def implied_vol(price, S, K, T, r, q, *, call: bool) -> float:
    # Calculate the implied volatility of an option given the price
    intrinsic = max(S - K, 0.0) if call else max(K - S, 0.0)
    if price < intrinsic - 1e-10:
        return float("nan")
    if price <= intrinsic + 1e-12:
        return 0.0

    def f(sig: float) -> float:
        return bs_price(S, K, T, r, q, sig, call=call) - price

    lo, hi = 1e-6, 5.0
    fl, fh = f(lo), f(hi)
    if fl * fh > 0:
        return float("nan")
    return brentq(f, lo, hi)


### Implied volatility by strike (calls vs puts)

**Implied vol** is the σ that reproduces the observed mid under your chosen `S`, `r`, `q`, and `T`. Plotting **call IV and put IV** on the same axes shows the **smile/skew** and any **call/put disagreement** at the same strike (often due to liquidity differences, dividends or microstructure noise).

In [7]:
# Implied vols from listed prices (same S as parity median)
option_prices["IV_Call"] = [
    implied_vol(c, S_median, k, T, r, q, call=True)
    for c, k in zip(option_prices["Calls"], option_prices["Strike"])
]
option_prices["IV_Put"] = [
    implied_vol(p, S_median, k, T, r, q, call=False)
    for p, k in zip(option_prices["Puts"], option_prices["Strike"])
]

forward = S_median * np.exp((r - q) * T)
calls_df = option_prices[np.isfinite(option_prices["IV_Call"])].copy()
puts_df = option_prices[np.isfinite(option_prices["IV_Put"])].copy()

fig = go.Figure(
    data=[
        go.Scatter(
            x=calls_df["Strike"],
            y=calls_df["IV_Call"] * 100,
            mode="lines+markers",
            name="Calls",
            hovertemplate="Strike=%{x}<br>IV=%{y:.2f}%<extra></extra>",
        ),
        go.Scatter(
            x=puts_df["Strike"],
            y=puts_df["IV_Put"] * 100,
            mode="lines+markers",
            name="Puts",
            hovertemplate="Strike=%{x}<br>IV=%{y:.2f}%<extra></extra>",
        ),
    ]
)
fig.add_vline(
    x=forward,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Forward {forward:.2f}",
    annotation_position="top",
)
fig.update_layout(
    title="Implied volatility vs strike",
    xaxis_title="Strike",
    yaxis_title="Implied vol (%)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
fig.show()

### Delta

BS Delta calculation

In [8]:
# Deltas using implied vol at each strike (same S as IV)
option_prices["Delta_Call"] = [
    bs_delta(S_median, k, T, r, q, sig, call=True) if np.isfinite(sig) else np.nan
    for sig, k in zip(option_prices["IV_Call"], option_prices["Strike"])
]
option_prices["Delta_Put"] = [
    bs_delta(S_median, k, T, r, q, sig, call=False) if np.isfinite(sig) else np.nan
    for sig, k in zip(option_prices["IV_Put"], option_prices["Strike"])
]
d_calls = option_prices[np.isfinite(option_prices["Delta_Call"])].copy()
d_puts = option_prices[np.isfinite(option_prices["Delta_Put"])].copy()
fig_delta = go.Figure(
    data=[
        go.Scatter(
            x=d_calls["Strike"],
            y=d_calls["Delta_Call"],
            mode="lines+markers",
            name="Call Δ",
            hovertemplate="Strike=%{x}<br>Δ=%{y:.4f}<extra></extra>",
        ),
        go.Scatter(
            x=d_puts["Strike"],
            y=d_puts["Delta_Put"],
            mode="lines+markers",
            name="Put Δ",
            hovertemplate="Strike=%{x}<br>Δ=%{y:.4f}<extra></extra>",
        ),
    ]
)
fig_delta.add_vline(
    x=forward,
    line_dash="dash",
    line_color="gray",
    annotation_text=f"Forward {forward:.2f}",
    annotation_position="top",
)
fig_delta.update_layout(
    title="Black–Scholes delta vs strike (σ = implied vol per strike)",
    xaxis_title="Strike",
    yaxis_title="Delta",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
)
fig_delta.show()


Generate the Option Chain combining prices, vols and deltas

In [ ]:
# Options chain
options_chain = pd.DataFrame(
    {
        "Call_Price": option_prices["Calls"].to_numpy(),
        "Call_Vol": option_prices["IV_Call"].to_numpy() * 100,
        "Call_Delta": option_prices["Delta_Call"].to_numpy(),
        "Put_Delta": option_prices["Delta_Put"].to_numpy(),
        "Put_Vol": option_prices["IV_Put"].to_numpy() * 100,
        "Put_Price": option_prices["Puts"].to_numpy(),
    },
    index=pd.Index(option_prices["Strike"], name="Strike"),
)
options_chain.attrs.update(
    {
        "underlying": "AAPL",
        "valuation_date": valuation_date.isoformat(),
        "expiry": expiry_date.isoformat(),
        "T_years": float(T),
        "spot": float(S0),
        "S_parity_median": float(S_median),
        "forward": float(forward),
        "r": float(r),
        "q": float(q),
    }
)
pd.Series(options_chain.attrs, name="chain_meta")
options_chain

,C_last,C_IV_pct,C_delta,P_delta,P_IV_pct,P_last
Strike,,,,,,
200,63.99,45.595159,0.928126,-0.047598,39.665445,0.95
205,58.95,41.863606,0.925239,-0.058470,38.463371,1.17
210,54.35,40.773235,0.910062,-0.072277,37.410792,1.46
215,49.74,39.294165,0.894181,-0.088753,36.336263,1.81
220,45.18,37.766407,0.875818,-0.108026,35.188559,2.22
225,40.74,36.431823,0.853178,-0.131319,34.100138,2.73
230,36.39,35.078492,0.826831,-0.158940,33.024697,3.35
235,32.19,33.844236,0.795437,-0.191622,31.989923,4.11
240,28.13,32.609284,0.759152,-0.229565,30.945206,5.02


### Vertical spreads: call spreads and put spreads

Screening for optimal call spreads to buy based on metrics of leveraged payoff and discount to buying the outright long call only.

Rank call/put spreads by discount to outright call or put, and max payoff/cost.

When buying a call spread, we want a large discount (at least 30%), and the highest payoff/cost possible. But there is a measure missing: how "far" the long strike is, in number of sigma of realised vol.

In [12]:
# Vertical spreads only (positive net price).
# Call: long K_low, short K_high (K_low < K_high). Outright = long call at K_low only.
#   discount vs outright (%) = 100 * C(K_high) / C(K_low)
#   max payoff at expiry (S >= K_high) = K_high - K_low; cost = C(K_low) - C(K_high)
# Put: long K_high, short K_low. Outright = long put at K_high only.
#   discount vs outright (%) = 100 * P(K_low) / P(K_high)
#   max payoff (S <= K_low) = K_high - K_low; cost = P(K_high) - P(K_low)

K = option_prices["Strike"].to_numpy(dtype=float)
C = option_prices["Calls"].to_numpy(dtype=float)
P = option_prices["Puts"].to_numpy(dtype=float)
n = len(K)

# constraints:
# strike spread >= 15
# discount >= 25%
min_strike_spread = 15
min_discount = 25

call_x, call_y, call_text = [], [], []
call_records = []
for i in range(n):
    for j in range(i + 1, n):
        k_lo, k_hi = K[i], K[j]
        # eliminate tight spreads
        if k_hi - k_lo < min_strike_spread:
            continue
        c_lo, c_hi = C[i], C[j]
        cost = c_lo - c_hi
        if cost <= 1e-12:
            continue
        width = k_hi - k_lo
        if c_lo <= 1e-12:
            continue
        discount_pct = 100.0 * c_hi / c_lo
        # eliminate low discounts
        if discount_pct < min_discount:
            continue
        call_x.append(discount_pct)
        call_y.append(width / cost)
        call_text.append(f"long {k_lo:g} / short {k_hi:g}")
        call_records.append(
            {
                "K_long": k_lo,
                "K_short": k_hi,
                "strike_spread": width,
                "discount_pct": discount_pct,
                "net_debit": cost,
                "max_payoff": width,
                "max_payoff_over_cost": width / cost,
            }
        )

put_x, put_y, put_text = [], [], []
for i in range(n):
    for j in range(i + 1, n):
        k_lo, k_hi = K[i], K[j]
        p_lo, p_hi = P[i], P[j]
        cost = p_hi - p_lo
        if cost <= 1e-12:
            continue
        width = k_hi - k_lo
        if p_hi <= 1e-12:
            continue
        discount_pct = 100.0 * p_lo / p_hi
        put_x.append(discount_pct)
        put_y.append(width / cost)
        put_text.append(f"long {k_hi:g} / short {k_lo:g}")

fig_spreads = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Call vertical spreads (debit)",
        "Put vertical spreads (debit)",
    ),
    horizontal_spacing=0.08,
)
fig_spreads.add_trace(
    go.Scatter(
        x=call_x,
        y=call_y,
        mode="markers",
        marker=dict(size=8, opacity=0.7),
        text=call_text,
        hovertemplate=(
            "%{text}<br>discount vs outright=%{x:.1f}%<br>max payoff/cost=%{y:.2f}<extra></extra>"
        ),
        name="Call spreads",
    ),
    row=1,
    col=1,
)
fig_spreads.add_trace(
    go.Scatter(
        x=put_x,
        y=put_y,
        mode="markers",
        marker=dict(size=8, opacity=0.7),
        text=put_text,
        hovertemplate=(
            "%{text}<br>discount vs outright=%{x:.1f}%<br>max payoff/cost=%{y:.2f}<extra></extra>"
        ),
        name="Put spreads",
    ),
    row=1,
    col=2,
)
fig_spreads.update_xaxes(title_text="Discount vs outright (%)", row=1, col=1)
fig_spreads.update_xaxes(title_text="Discount vs outright (%)", row=1, col=2)
fig_spreads.update_yaxes(title_text="Max payoff / cost", row=1, col=1)
fig_spreads.update_yaxes(title_text="Max payoff / cost", row=1, col=2)
fig_spreads.update_layout(
    template="plotly_white",
    showlegend=False,
    title_text="Vertical spreads: discount vs outright (%) vs payoff multiple",
    title_x=0.5,
    height=480,
)
fig_spreads.show()

call_spreads_all = pd.DataFrame(call_records)
call_spreads_filtered = (
    call_spreads_all[
        (call_spreads_all["discount_pct"] > 30)
        & (call_spreads_all["strike_spread"] >= 20)
    ]
    .sort_values("max_payoff_over_cost", ascending=False)
    .reset_index(drop=True)
)
call_spreads_filtered.head(10)

,K_long,K_short,strike_spread,discount_pct,net_debit,max_payoff,max_payoff_over_cost
0,275.0,295.0,20.0,30.647482,4.82,20.0,4.149378
1,270.0,290.0,20.0,32.513966,6.04,20.0,3.311258
2,265.0,285.0,20.0,34.683099,7.42,20.0,2.695418
3,260.0,280.0,20.0,37.393768,8.84,20.0,2.262443
4,255.0,280.0,25.0,30.697674,11.92,25.0,2.097315
5,255.0,275.0,20.0,40.406977,10.25,20.0,1.951220
6,250.0,275.0,25.0,33.737864,13.65,25.0,1.831502
7,250.0,270.0,20.0,43.446602,11.65,20.0,1.716738
8,245.0,270.0,25.0,36.907216,15.30,25.0,1.633987
9,240.0,270.0,30.0,31.816566,19.18,30.0,1.564129


In [ ]:
# show call spread with max payoff/cost
params = call_spreads_filtered.iloc[0]

print("Trade recommendation:")
print(f"long {int(params['K_long'])} / short {int(params['K_short'])}")
print(f"discount vs outright={params['discount_pct']:.1f}%")
print(f"max payoff/cost={params['max_payoff_over_cost']:.2f}")
print(f"max payoff={int(params['max_payoff'])}")
print(f"cost={params['net_debit']}")


Trade recommendation:
long 275 / short 295
discount vs outright=30.6%
max payoff/cost=4.15
max payoff=20
cost=4.82
